# Practice 4-3. 가상 문서 임베딩 (HyDE, Hypothetical Document Embeddings)

질문을 이용해 검색하는 대신, 질문에 대한 '가상의 답변 문서'를 LLM으로 먼저 생성한 뒤 그 가상 문서로 유사도 검색을 수행하는 질의 변형 기법입니다.
책은 OpenAI(gpt-4o)를 사용하지만, 이 노트북은 LM Studio 로컬 LLM과 Chroma 서버로 동일한 파이프라인을 구성합니다.

## 0. 환경 설정

In [ ]:
import re
import chromadb

# --- LM Studio ---
LMSTUDIO_BASE_URL = "http://...:.../v1"         # /v1 까지 포함
LMSTUDIO_API_KEY  = "lm-studio"                              # LM Studio는 키 검증 안 함 (더미)

EMBED_MODEL = "text-embedding-qwen3-embedding-8b"
LLM_MODEL   = "qwen3-30b-a3b-instruct-2507"   # LM Studio에 로드되어 있는 로컬 채팅 모델

# --- Chroma 서버 ---
CHROMA_HOST = "chromadb"
CHROMA_PORT = 8000
COLLECTION  = "practice4_hyde"

# --- 입력 파일 (노트북 기준 상대경로) ---
DATA_PATH = "How_to_invest_money.txt"


def strip_think(text: str) -> str:
    """추론 모델의 <think>...</think> 블록 제거."""
    if text is None:
        return ""
    return re.sub(r"(^|<think>).*?</think>", "", text, flags=re.DOTALL).strip()


chroma_client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
print("Chroma 서버 연결:", chroma_client.heartbeat())

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 임베딩 모델
embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    check_embedding_ctx_length=False,
    chunk_size=8,
)

# 채팅 모델 (가상 문서 생성 + 최종 답변 생성용)
# LM Studio의 로컬 모델은 추론(사고) 모델이라 답변 전에 reasoning 토큰을 소모한다.
# 가상 문서 생성처럼 다소 긴 응답을 요구하면 reasoning만으로 2000토큰 이상 소모되어
# max_tokens=2048에서는 답변이 빈 문자열로 잘리는 경우가 있어 4096으로 넉넉히 잡는다.
llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    temperature=0,
    max_tokens=4096,
)

print("EMBED:", len(embeddings.embed_query("연결 테스트")), "차원")
print("LLM  :", strip_think(llm.invoke("연결 테스트. 'ok' 라고만 답하세요.").content)[:50])

## 1. 문서 로드 · 분할 · Chroma 벡터스토어 구축

`How_to_invest_money.txt`를 로드한 뒤 1000자 단위(중복 200자)로 분할하고, Chroma 서버에 임베딩하여 저장합니다.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# 문서 로더 설정 (encoding="utf-8-sig" : 원본 파일 선두의 BOM 제거)
loaders = [TextLoader(DATA_PATH, encoding="utf-8-sig")]

docs = []
for loader in loaders:
    docs.extend(loader.load())

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = recursive_splitter.split_documents(docs)
print("분할된 문서 수:", len(split_docs))

# 재실행 안전: 이전 실행에서 남은 컬렉션 제거 후 새로 생성
try:
    chroma_client.delete_collection(COLLECTION)
except Exception:
    pass

vectorstore = Chroma(client=chroma_client, collection_name=COLLECTION, embedding_function=embeddings)
vectorstore.add_documents(split_docs)
print("벡터스토어 저장 완료:", vectorstore._collection.count(), "개")

retriever = vectorstore.as_retriever()

## 2. HyDE 파이프라인 구성

HyDE 방식은 각각의 기능을 수행하는 여러 체인을 순차적으로 연결하는 방식입니다. 다음과 같은 체인을 만든 후 결합합니다.

- 가상 문서 생성 체인
- 문서 검색 체인
- 최종 응답 생성 체인

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda


# 1. 가상 문서 생성 체인
def create_virtual_doc_chain():
    system = "당신은 고도로 숙련된 AI입니다."
    user = """
주어진 질문 '{query}'에 대해 직접적으로 답변하는 가상의 문서를 생성하세요.
문서의 크기는 {chunk_size} 글자 언저리여야 합니다.
"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", system),
        ("human", user),
    ])
    return prompt | llm | StrOutputParser()


# 2. 문서 검색 체인 (가상 문서로 벡터 DB에서 유사 문서 검색)
# langchain 1.x: retriever.get_relevant_documents 대신 invoke 사용
def create_retrieval_chain():
    return RunnableLambda(lambda x: retriever.invoke(x["virtual_doc"]))


# 유틸리티 함수: 검색된 문서에서 순수 텍스트만 추출
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# 3. 최종 응답 생성 체인
def create_final_response_chain():
    final_prompt = ChatPromptTemplate.from_template("""
다음 정보와 질문을 바탕으로 답변해주세요.
컨텍스트: {context}
질문: {question}
답변:
""")
    return final_prompt | llm

## 3. 로깅 유틸리티 및 전체 파이프라인 구성

각 단계에서 입력과 출력을 로깅하는 유틸리티 함수를 만들고, 앞서 정의한 3개의 체인을 `RunnableLambda`로 감싸 `virtual_doc_step → retrieval_step → context_formatting_step → final_response_step` 순으로 연결합니다.

In [ ]:
def print_input_output(input_data, output_data, step_name):
    print(f"\n--- {step_name} ---")
    print(f"Input: {str(input_data)[:200]}")
    print(f"Output: {str(output_data)[:300]}")
    print("-" * 50)


def create_pipeline_with_logging():
    virtual_doc_chain = create_virtual_doc_chain()
    retrieval_chain = create_retrieval_chain()
    final_response_chain = create_final_response_chain()

    # 가상 문서 생성 단계
    def virtual_doc_step(x):
        result = {"virtual_doc": virtual_doc_chain.invoke({
            "query": x["question"],
            "chunk_size": 200,
        })}
        print_input_output(x, result, "Virtual Doc Generation")
        return {**x, **result}

    # 문서 검색 단계
    def retrieval_step(x):
        result = {"retrieved_docs": retrieval_chain.invoke(x)}
        print_input_output(x, result, "Document Retrieval")
        return {**x, **result}

    # 컨텍스트 포매팅 단계
    def context_formatting_step(x):
        result = {"context": format_docs(x["retrieved_docs"])}
        print_input_output(x, result, "Context Formatting")
        return {**x, **result}

    # 최종 응답 생성 단계
    def final_response_step(x):
        result = final_response_chain.invoke(x)
        print_input_output(x, result, "Final Response Generation")
        return result

    # 전체 파이프라인 구성
    pipeline = (
        RunnableLambda(virtual_doc_step)
        | RunnableLambda(retrieval_step)
        | RunnableLambda(context_formatting_step)
        | RunnableLambda(final_response_step)
    )

    return pipeline


# 파이프라인 객체 생성
pipeline = create_pipeline_with_logging()
print("pipeline 준비 완료")

## 4. 질문과 답변 예

`pipeline.invoke()`에 전달하는 입력값은 반드시 `{"question": question}` 형태의 딕셔너리여야 합니다. 이는 파이프라인의 첫 단계인 `virtual_doc_step` 함수가 `question` 키를 기준으로 동작하기 때문입니다.

In [ ]:
# 질문과 답변 예
question = "주식 시장의 변동성이 높을 때 투자 전략은 무엇인가요?"
response = pipeline.invoke({"question": question})
print(f"\n\n최종 답변: {strip_think(response.content)}")